### This notebook creates the job that automates running the SQL and R notebooks
First we need to load (and install if not present) packages

In [0]:
%r
    library(tidyverse)

    install.packages("devtools")
    library(devtools)

    install_github("databrickslabs/databricks-sdk-r")
    library(databricks)

Create a widget that stores your personal access token (PAT) that you have used to connect to databricks. If you have completed the XXX step, this will be stored in your .Renviron file (if you haven't noted it down anywhere else). 

**IMPORTANT NOTE: Do not include your personal access token in the code.** We use a text widget to specify our PAT to ensure we don't expose it. If it were coded in the notebook then anyone with access to the notebook would be able to use it to emulate our Databricks access permissions. Databricks has in-built protections for this, so it should throw an error if you try to push a commit that contains the token (either in the code or saved as a widget), so any references in the code or the widget value box must be removed before committing to a remote repository.

In [0]:
%py
dbutils.widgets.text("api_token", "")

### Connect to databricks API
Connect to the Databricks API using the databricks package and get cluster details needed to run the job. The cluster_id variable must only contain one value - here we're automating this step by filtering to the Personal Compute Cluster, as this is the one that we will need to use to run R cells (which you should have set up as an R-Studio Compute - might need to add some details earlier). 

In [0]:
%r
host <- "https://adb-5037484389568426.6.azuredatabricks.net/" 

api_token <- dbutils.widgets.get("api_token")  

client <- databricks::DatabricksClient(host = host, token = api_token)  

clusters <- databricks::clustersList(client)  

# Get cluster variables
clusters <- clusters %>% 
       select(cluster_id, 
              creator_user_name, 
              cluster_name, 
              spark_version, 
              cluster_cores, 
              cluster_memory_mb) 

# Filter to your Personal Compute Cluster 
# (in this case I named my R-Studio cluster so that this step could be automated, filtering)
personal_compute_cluster <- clusters %>%
       filter(str_detect(cluster_name, "RStudio"))
        
# Get your Personal Compute Cluster ID
cluster_id <- personal_compute_cluster %>%
       select(cluster_id) %>%
       toString()

# Check that there is only one output, and confirm that it is the right cluster
print(personal_compute_cluster)

### Define parameters of the job that we want to run
as above

In [0]:
%r
job_name <- "Average pupil premium pipeline" 

sql_notebook_path <- "/Workspace/Users/robert.tarrant@education.gov.uk/techskills_workshops/Databricks_workshops/Databricks workshop/03 Migrating existing processes/hybrid_pipeline_solutions/1_databricks_job_solution/source_code/1_sql_code.ipynb" 
r_notebook_path <- "/Workspace/Users/robert.tarrant@education.gov.uk/techskills_workshops/Databricks_workshops/Databricks workshop/03 Migrating existing processes/hybrid_pipeline_solutions/1_databricks_job_solution/source_code/2_r_code.ipynb" 

task_key_sql <- "sql_key" 
task_key_r <- "r_key"

### SQL task

In [0]:
%r
# First the SQL job
sql_job_task <- list(
 task_key = task_key_sql,
 existing_cluster_id = cluster_id,
 notebook_task = list(
                  notebook_path = sql_notebook_path,
                  source = "WORKSPACE"
                  )
 )
 
# Then the R job
r_job_task <- list(
 task_key = task_key_r,
 existing_cluster_id = cluster_id,
 notebook_task = list(
                  notebook_path = r_notebook_path,
                  source = "WORKSPACE"
                  ),
  depends_on = list(task_key = task_key_sql),
  run_if = "ALL_SUCCESS"
 )

### Create the actual job

In [0]:
%r
workflow <- jobsCreate(client,
                       name = job_name,
                       tasks = list(sql_job_task, #list
                                    r_job_task), #list
                       email_notifications = list(
                         on_success = c("robert.tarrant@education.gov.uk") # <-- please make sure to update to your email!
                         )
                       )

### Run the job
Start executing the job by running the cell below, and check on it's progress by using the links generated in the following cells.

In [0]:
%r
job_id <- workflow$job_id  

job_run <- jobsRunNow(client,
                      job_id = job_id)

job_run_id <- job_run$run_id

job_link <- paste0(host,"jobs/",job_id)
job_run_link <- paste0(job_link,"/runs/", job_run_id)

In [0]:
%r
# Show the job link to view the job (right click to open in a new tab)
job_link

In [0]:
%r
# Show the job link to view the job (right click to open in a new tab)
job_run_link